In [ ]:
# LLMS
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm = ChatGroq(model="openai/gpt-oss-120b")
light_llm = init_chat_model(
    model="meta-llama/llama-3.3-70b-instruct", 
    model_provider="openrouter",
)
reasoning_llm = init_chat_model(
    model="deepseek/deepseek-r1",
    model_provider="openrouter"
)


In [ ]:
from pageindex import PageIndexClient
from dotenv import load_dotenv
load_dotenv()

# 1. create pageindex client
pageindexClient = PageIndexClient(
    index="cloud", 
    chat="deepseek/deepseek-r1:free",
)


In [ ]:
print(pageindexClient.list_documents())

In [ ]:
# 2. upload and index page(upload pdf ->pageindex uses llm to read doc structure -> builds hierarchical tree index -> returns docId)

pdf_path = "AI Engineering Guidebook.pdf"
print(f"uploading{pdf_path}")
# res = pageindexClient.submit_document(pdf_path)
doc_id = pageindexClient.get_document_id(pdf_path)

print("Uploaded")
print(f"Doc Id {doc_id}")

In [ ]:
import json
# 3. Fetch the full tree
tree_res = pageindexClient.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_res.get("result", [])

print("tree_res: ", tree_res)
print("pageindex_tree: ", pageindex_tree)

print(f"top Level Sections : {len(pageindex_tree)}")
print("First Node")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=3))

In [ ]:
# Pretty print the whole tree

def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    
    for node in nodes:
        prefix = " "*indent + ("L " if indent > 0 else "")
        page = node.get("start_index", "?")

        print(f"{prefix}[{node['node_id']}] {node['title']} (p. {page})")

        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


print("Full Document Structure:\n")
print_tree(pageindex_tree, 6)

In [ ]:
# Count the total number of nodes in a tree

def count_nodes(nodes):
    total = len(nodes)
    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total
print("Total Nodes in a tree ")
count_nodes(pageindex_tree)

In [ ]:
import os
from openai import OpenAI

# 1. Initialize the client with Groq's endpoint and your Groq API key
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY") 
)



In [ ]:
import json


# LLM Tree Search (for a query it evaluates page tree (DOcument) structure to return the relevant node id's for subsequent reterival)
def llm_search_tree(query, tree, model) -> dict:
    """
    Sends query + doc tree to llm 
    llm reasons over structure and returns reterival nodes
    """
    def compress(nodes):
        out = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                # "page_index": node.get("page_index", "?"),
                # "summary": node.get("text", "")[:50]
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.

    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, separators=(",", ":"))}

    Reply ONLY in this exact JSON format:

   {{
        "node_list": ["node_id1", "node_id2"]
    }}
    """
    # 2. Submit the chat completion request
    response = client.chat.completions.create(
        model=model, 
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)
    




In [ ]:
# Perform a sample test

query = "what is context engineering"
model="qwen/qwen3.8-27b"


print(f"Query : {query}")

res = llm_search_tree(query, pageindex_tree, model)
print(f"LLM Reasoning ...")
print(res.get("thinking", "N/A"))
print("Selected Node Ids\n\n")
print(res.get("node_list", []))

In [ ]:
# Find Nodes by id
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    # Convert to a set for O(1) lightning-fast lookups
    target_set = set(target_ids) 
    
    for node in tree:
        if node["node_id"] in target_set:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

reterived_nodes = find_nodes_by_ids(pageindex_tree, res.get("node_list", []))

In [ ]:
# 
def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b"):
    if not nodes:
        print("No relevant Sections found")
        return
    # build context string from reterived nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[section: {node['title']} | Page. {node.get('page_index', node.get('start_index', '?'))}]\n"
            f"{node.get('text')}"
        )
    context = "\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parentheses.
    Be concise and precise.

    Question: {query}

    Context:
    {context}
    Answer: """
    # 2. Submit the chat completion request
    response = client.chat.completions.create(
        model=model, 
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


In [ ]:
import json

# 1. Dynamically pull the exact active models available on your account right now
try:
    available_models = [m.id for m in client.models.list().data]
    # Filter out whisper audio models or specific embeddings models if any exist
    text_models = [m for m in available_models if "whisper" not in m.lower() and "embed" not in m.lower()]
    
    # Grab the first available production text model slug automatically
    active_model = text_models[0] if text_models else "llama-3.3-70b-versatile"
    print(f"✅ Found active Groq text models: {text_models}")
    print(f"🚀 Using dynamic model target: '{active_model}'")
except Exception as e:
    # Fallback if the client list endpoint hits a rate block
    active_model = "llama-3.3-70b-versatile" 
    print(f"⚠️ Could not list models dynamically ({e}). Falling back to baseline slug.")



In [ ]:
generate_answer(query, reterived_nodes, "openai/gpt-oss-120b")

In [ ]:
answer = pageindexClient.chat(
    "What is context engineering?",
    doc_id=doc_id
)

print(answer)